In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('proact_preprocessed_S1.csv')
df.head()

,subject_id,ALSFRS_Delta,Q1_Speech,Q2_Salivation,Q3_Swallowing,Q4_Handwriting,Q5_Cutting,Q6_Dressing_and_Hygiene,Q7_Turning_in_Bed,Q8_Walking,Q9_Climbing_Stairs,R_1_Dyspnea,R_2_Orthopnea,R_3_Respiratory_Insufficiency,Age_Base,Sex_Female,Treatment_Active,Bulbar_Onset,FVC_Base,BMI_Base
0,16497,0.0,4.0,4.0,4.0,3.0,3.0,2.0,1.0,1.0,0.0,4.0,4.0,4.0,0.016919,1.0,1.0,0.0,0.126976,-0.356485
1,53215,8.0,3.0,2.0,4.0,4.0,4.0,3.0,3.0,3.0,3.0,4.0,4.0,4.0,-1.013892,0.0,1.0,1.0,0.289065,-0.354520
2,53215,71.0,3.0,2.0,4.0,3.0,4.0,3.0,3.0,3.0,1.0,4.0,4.0,4.0,-1.013892,0.0,1.0,1.0,0.289065,-0.354520
3,53215,140.0,3.0,3.0,4.0,3.0,4.0,2.0,3.0,3.0,3.0,3.0,4.0,4.0,-1.013892,0.0,1.0,1.0,0.289065,-0.354520
4,53215,204.0,3.0,4.0,3.0,3.0,3.0,3.0,3.0,2.0,2.0,4.0,4.0,4.0,-1.013892,0.0,1.0,1.0,0.289065,-0.354520


In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from factor_analyzer import FactorAnalyzer
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. DISCRETIZE TIME AND CREATE LAGS
# ==========================================
# Sort to ensure chronological order per patient
df = df.sort_values(by=['subject_id', 'ALSFRS_Delta'])

# Create a lagged time variable to calculate the exact gap between discrete visits
df['Delta_Lag'] = df.groupby('subject_id')['ALSFRS_Delta'].shift(1)
df['Time_Since_Last_Visit'] = df['ALSFRS_Delta'] - df['Delta_Lag']

# ==========================================
# 2. SCENARIOS & COVARIATES
# ==========================================
domains = {
    'Bulbar': ['Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing'],
    'Fine_Motor': ['Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene'],
    'Gross_Motor': ['Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs'],
    'Respiratory': ['R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency']
}
all_items = [item for sublist in domains.values() for item in sublist]

# Notice ALSFRS_Delta is replaced by Time_Since_Last_Visit for the discrete-time step
covariates = 'Time_Since_Last_Visit + Sex_Female + Treatment_Active + Age_Base + Bulbar_Onset + FVC_Base + BMI_Base'

scenarios = {'Total Score': all_items, **domains}
results_list = []
gkf = GroupKFold(n_splits=5)

print("Starting Discrete-Time Factor Analysis Pipeline...\n")

# ==========================================
# 3. EVALUATION LOOP
# ==========================================
for target_name, items in scenarios.items():
    print(f"Fitting Discrete-Time AR(1) FA for {target_name}...")
    
    # --- Step A: Measurement Model (Factor Analysis) ---
    # Fit a 1-factor model to the items to extract the continuous latent trait
    fa = FactorAnalyzer(n_factors=1, rotation=None)
    fa.fit(df[items])
    
    # Extract factor scores and parameters for later back-transformation
    factor_scores = fa.transform(df[items])
    df['Factor'] = factor_scores[:, 0]
    loadings = fa.loadings_[:, 0] 
    means = df[items].mean().values
    
    # --- Step B: Create Discrete Autoregressive Lags ---
    df['Factor_Lag'] = df.groupby('subject_id')['Factor'].shift(1)
    
    # We must drop the very first visit (t=0) because a Markov/AR model 
    # requires a previous state to predict the current state.
    df_dynamic = df.dropna(subset=['Factor_Lag', 'Time_Since_Last_Visit'])
    
    # --- Step C: Full Model Fit for AIC/BIC ---
    formula = f"Factor ~ Factor_Lag + {covariates}"
    try:
        full_model = smf.mixedlm(formula, data=df_dynamic, groups=df_dynamic["subject_id"]).fit(disp=False)
        aic, bic = full_model.aic, full_model.bic
    except:
        aic, bic = np.nan, np.nan

    # --- Step D: Cross-Validation for Out-of-Sample RMSE ---
    item_rmses = {item: [] for item in items}
    aggregate_rmses = []
    
    for train_idx, test_idx in gkf.split(df_dynamic, groups=df_dynamic['subject_id']):
        train_df, test_df = df_dynamic.iloc[train_idx], df_dynamic.iloc[test_idx]
        
        try:
            # 1. Train the discrete-time LMM on the latent factor
            cv_model = smf.mixedlm(formula, data=train_df, groups=train_df["subject_id"]).fit(disp=False)
            
            # 2. Predict the next factor state for unseen data
            pred_factor = cv_model.predict(exog=test_df)
            
            # 3. Map the predicted factor back to raw expected item scores
            # Equation: X_hat = (F_hat * Loading) + Mean
            expected_aggregate_score = np.zeros(len(test_df))
            
            for i, item in enumerate(items):
                exp_item_score = (pred_factor * loadings[i]) + means[i]
                
                # Clip bounds to mimic the strict limits of the ALSFRS-R scale
                exp_item_score = np.clip(exp_item_score, 0, 4)
                
                rmse_item = np.sqrt(mean_squared_error(test_df[item], exp_item_score))
                item_rmses[item].append(rmse_item)
                expected_aggregate_score += exp_item_score
                
            # Aggregate RMSE
            actual_aggregate = test_df[items].sum(axis=1)
            rmse_agg = np.sqrt(mean_squared_error(actual_aggregate, expected_aggregate_score))
            aggregate_rmses.append(rmse_agg)
            
        except Exception:
            continue
            
    # --- Step E: Store Results ---
    results_list.append({
        'Level': 'Overall' if target_name == 'Total Score' else 'Domain',
        'Target': target_name,
        'Out-of-Sample RMSE': np.mean(aggregate_rmses),
        'Latent_AIC': aic,
        'Latent_BIC': bic
    })
    
    if target_name != 'Total Score':
        for item in items:
            results_list.append({
                'Level': 'Individual Item',
                'Target': item,
                'Out-of-Sample RMSE': np.mean(item_rmses[item]),
                'Latent_AIC': np.nan, 
                'Latent_BIC': np.nan
            })

# ==========================================
# 4. PRINT RESULTS
# ==========================================
evaluation_df = pd.DataFrame(results_list)
evaluation_df['Level'] = pd.Categorical(evaluation_df['Level'], categories=['Individual Item', 'Domain', 'Overall'], ordered=True)
evaluation_df = evaluation_df.sort_values(['Level', 'Target'])

print("\n" + "="*85)
print("DISCRETE-TIME FACTOR ANALYSIS: OUT-OF-SAMPLE PREDICTIVE ACCURACY")
print("="*85)
format_dict = {'Out-of-Sample RMSE': '{:.3f}'.format, 'Latent_AIC': '{:.1f}'.format, 'Latent_BIC': '{:.1f}'.format}
print(evaluation_df.to_string(formatters=format_dict, index=False))

Starting Discrete-Time Factor Analysis Pipeline...

Fitting Discrete-Time AR(1) FA for Total Score...
Fitting Discrete-Time AR(1) FA for Bulbar...
Fitting Discrete-Time AR(1) FA for Fine_Motor...
Fitting Discrete-Time AR(1) FA for Gross_Motor...
Fitting Discrete-Time AR(1) FA for Respiratory...

DISCRETE-TIME FACTOR ANALYSIS: OUT-OF-SAMPLE PREDICTIVE ACCURACY
          Level                        Target Out-of-Sample RMSE Latent_AIC Latent_BIC
Individual Item                     Q1_Speech              0.570        NaN        NaN
Individual Item                 Q2_Salivation              0.569        NaN        NaN
Individual Item                 Q3_Swallowing              0.527        NaN        NaN
Individual Item                Q4_Handwriting              0.707        NaN        NaN
Individual Item                    Q5_Cutting              0.640        NaN        NaN
Individual Item       Q6_Dressing_and_Hygiene              0.742        NaN        NaN
Individual Item             Q

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error
from girth import grm_mml
from scipy.special import expit
from factor_analyzer import FactorAnalyzer
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. SETUP AND COVARIATES
# ==========================================
# Make sure your original 'df' is loaded
all_items = [
    'Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing', 'Q4_Handwriting',
    'Q5_Cutting', 'Q6_Dressing_and_Hygiene', 'Q7_Turning_in_Bed',
    'Q8_Walking', 'Q9_Climbing_Stairs', 'R_1_Dyspnea',
    'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency'
]

# Standard Covariates
covariates = 'ALSFRS_Delta + Sex_Female + Treatment_Active + Age_Base'
# Note: Adjust the covariates string to exactly match what you used in your final MCMC run

results_list = []
print("Calculating In-Sample Baseline Metrics...\n")



# ==========================================
# BASELINE 3: DISCRETE-TIME FACTOR ANALYSIS
# ==========================================
print("3. Fitting Discrete-Time Factor Analysis...")
# Setup discrete lags
df = df.sort_values(by=['subject_id', 'ALSFRS_Delta'])
df['Delta_Lag'] = df.groupby('subject_id')['ALSFRS_Delta'].shift(1)
df['Time_Since_Last_Visit'] = df['ALSFRS_Delta'] - df['Delta_Lag']

# Fit Factor Analysis
fa = FactorAnalyzer(n_factors=1, rotation=None)
fa.fit(df[all_items])
df['Factor'] = fa.transform(df[all_items])[:, 0]
loadings = fa.loadings_[:, 0]
means = df[all_items].mean().values

# Create lagged targets
df['Factor_Lag'] = df.groupby('subject_id')['Factor'].shift(1)
df_dynamic = df.dropna(subset=['Factor_Lag', 'Time_Since_Last_Visit']).copy()

# Fit AR(1) LMM
covariates_ar = 'Time_Since_Last_Visit + Sex_Female + Treatment_Active + Age_Base'
formula_ar = f"Factor ~ Factor_Lag + {covariates_ar}"
ar_lmm = smf.mixedlm(formula_ar, data=df_dynamic, groups=df_dynamic["subject_id"]).fit(reml=False, disp=False)

# Map fitted factors back to expected raw scores
expected_total_ar = np.zeros(len(df_dynamic))
for i, item in enumerate(all_items):
    exp_score = (ar_lmm.fittedvalues * loadings[i]) + means[i]
    exp_score = np.clip(exp_score, 0, 4) # Enforce boundaries
    expected_total_ar += exp_score

actual_total_ar = df_dynamic[all_items].sum(axis=1)
rmse_ar = np.sqrt(mean_squared_error(actual_total_ar, expected_total_ar))
results_list.append({'Model': '3. Discrete-Time FA', 'Target': 'Total Score', 'In-Sample RMSE': rmse_ar})

# ==========================================
# FINAL OUTPUT
# ==========================================
# Add your OU process result manually to complete the table
results_list.append({'Model': '4. IRT + Latent OU Process', 'Target': 'Total Score', 'In-Sample RMSE': 1.410})

evaluation_df = pd.DataFrame(results_list)
print("\n" + "="*70)
print("FINAL ABLATION STUDY: IN-SAMPLE PREDICTIVE ACCURACY")
print("="*70)
print(evaluation_df.to_string(formatters={'In-Sample RMSE': '{:.3f}'.format}, index=False))

Calculating In-Sample Baseline Metrics...

3. Fitting Discrete-Time Factor Analysis...

FINAL ABLATION STUDY: IN-SAMPLE PREDICTIVE ACCURACY
                     Model      Target In-Sample RMSE
       3. Discrete-Time FA Total Score          3.405
4. IRT + Latent OU Process Total Score          1.410


In [6]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error
from factor_analyzer import FactorAnalyzer
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. SETUP
# ==========================================
domains = {
    'Bulbar': ['Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing'],
    'Fine_Motor': ['Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene'],
    'Gross_Motor': ['Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs'],
    'Respiratory': ['R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency']
}
all_items = [item for sublist in domains.values() for item in sublist]
scenarios = {'Total Score': all_items, **domains}

covariates_ar = 'Time_Since_Last_Visit + Sex_Female + Treatment_Active + Age_Base'
results_list = []

print("Extracting Discrete-Time FA Metrics...\n")

# ==========================================
# 1. PREPARE TIME LAGS
# ==========================================
df = df.sort_values(by=['subject_id', 'ALSFRS_Delta'])
df['Delta_Lag'] = df.groupby('subject_id')['ALSFRS_Delta'].shift(1)
df['Time_Since_Last_Visit'] = df['ALSFRS_Delta'] - df['Delta_Lag']

# ==========================================
# 2. FIT THE MODELS
# ==========================================
for target_name, items in scenarios.items():
    level = 'Overall' if target_name == 'Total Score' else 'Domain'
    print(f"Fitting {target_name}...")
    
    try:
        # Fit Factor Analysis
        fa = FactorAnalyzer(n_factors=1, rotation=None)
        fa.fit(df[items])
        df['Factor'] = fa.transform(df[items])[:, 0]
        loadings = fa.loadings_[:, 0]
        means = df[items].mean().values
        
        # Create lag for the AR(1) process
        df['Factor_Lag'] = df.groupby('subject_id')['Factor'].shift(1)
        df_dyn = df.dropna(subset=['Factor_Lag', 'Time_Since_Last_Visit']).copy()
        
        formula = f"Factor ~ Factor_Lag + {covariates_ar}"
        
        # --- ROBUST FALLBACK LOGIC ---
        try:
            model = smf.mixedlm(formula, data=df_dyn, groups=df_dyn["subject_id"])
            result = model.fit(method='lbfgs', reml=False, disp=False)
            fitted_factor = result.fittedvalues # This crashes if RE variance is 0
        except Exception as e:
            # If singular covariance, Mixed Model == OLS
            print(f"  -> LMM singular. Falling back to OLS (Random Effect Variance = 0).")
            ols_model = smf.ols(formula, data=df_dyn)
            result = ols_model.fit()
            fitted_factor = result.fittedvalues
        
        # Map predictions back to 0-4 scale
        expected_agg = np.zeros(len(df_dyn))
        for i, item in enumerate(items):
            exp_item = (fitted_factor * loadings[i]) + means[i]
            exp_item = np.clip(exp_item, 0, 4) # Enforce physical bounds
            expected_agg += exp_item
            
            # Save Individual Item metrics
            if level == 'Domain':
                rmse_item = np.sqrt(mean_squared_error(df_dyn[item], exp_item))
                results_list.append({'Level': 'Individual Item', 'Target': item, 'Discrete-Time FA RMSE': rmse_item})
                
        # Save Domain / Total Score metrics
        actual_agg = df_dyn[items].sum(axis=1)
        rmse_agg = np.sqrt(mean_squared_error(actual_agg, expected_agg))
        results_list.append({'Level': level, 'Target': target_name, 'Discrete-Time FA RMSE': rmse_agg})
        
    except Exception as e:
        print(f"  -> FAILED on {target_name}. Error: {e}")

# ==========================================
# 3. PRINT FORMATTED RESULTS
# ==========================================
eval_df = pd.DataFrame(results_list)
eval_df['Level'] = pd.Categorical(eval_df['Level'], categories=['Overall', 'Domain', 'Individual Item'], ordered=True)
eval_df = eval_df.sort_values(['Level', 'Target'])

print("\n" + "="*60)
print("MISSING METRICS: DISCRETE-TIME FACTOR ANALYSIS")
print("="*60)
print(eval_df.to_string(formatters={'Discrete-Time FA RMSE': '{:.3f}'.format}, index=False))

Extracting Discrete-Time FA Metrics...

Fitting Total Score...
  -> LMM singular. Falling back to OLS (Random Effect Variance = 0).
Fitting Bulbar...
  -> LMM singular. Falling back to OLS (Random Effect Variance = 0).
Fitting Fine_Motor...
  -> LMM singular. Falling back to OLS (Random Effect Variance = 0).
Fitting Gross_Motor...
  -> LMM singular. Falling back to OLS (Random Effect Variance = 0).
Fitting Respiratory...
  -> LMM singular. Falling back to OLS (Random Effect Variance = 0).

MISSING METRICS: DISCRETE-TIME FACTOR ANALYSIS
          Level                        Target Discrete-Time FA RMSE
        Overall                   Total Score                 3.525
         Domain                        Bulbar                 1.009
         Domain                    Fine_Motor                 1.395
         Domain                   Gross_Motor                 1.441
         Domain                   Respiratory                 1.284
Individual Item                     Q1_Speech     

In [11]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score
from girth import grm_mml
from scipy.special import expit
from factor_analyzer import FactorAnalyzer
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. SETUP & DATA PREP
# ==========================================
print("Loading data and setting up...")
# IMPORTANT: Load your actual patient data here
df = pd.read_csv('proact_preprocessed_S1.csv') 

domains = {
    'Bulbar': ['Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing'],
    'Fine_Motor': ['Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene'],
    'Gross_Motor': ['Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs'],
    'Respiratory': ['R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency']
}
all_items = [item for sublist in domains.values() for item in sublist]
scenarios = {'Total Score': all_items, **domains}

# Add aggregated true scores to dataframe
df['Total Score'] = df[all_items].sum(axis=1)
for dom, items in domains.items():
    df[dom] = df[items].sum(axis=1)

covariates = 'ALSFRS_Delta + Sex_Female + Treatment_Active + Age_Base'
covariates_ar = 'Time_Since_Last_Visit + Sex_Female + Treatment_Active + Age_Base'

results_list = []

def record_metrics(model_name, level, target, actual, expected, is_item=False):
    """Helper function to calculate and store all advanced metrics."""
    rmse = np.sqrt(mean_squared_error(actual, expected))
    mae = mean_absolute_error(actual, expected)
    
    if is_item:
        pred_cat = np.clip(np.round(expected), 0, 4)
        exact = np.mean(pred_cat == actual) * 100
        kappa = cohen_kappa_score(actual, pred_cat, weights='linear')
    else:
        exact, kappa = np.nan, np.nan
        
    results_list.append({
        'Model': model_name, 'Level': level, 'Target': target,
        'RMSE': rmse, 'MAE': mae, 'Exact Match %': exact, 'Weighted Kappa': kappa
    })

def get_irt_expected(theta, discrimination, thresholds):
    """Maps latent theta back to 0-4 scale."""
    p_ge = [expit(discrimination * (theta - th)) for th in thresholds]
    p4 = p_ge[3]
    p3 = p_ge[2] - p_ge[3]
    p2 = p_ge[1] - p_ge[2]
    p1 = p_ge[0] - p_ge[1]
    p0 = 1.0 - p_ge[0]
    return (0 * p0) + (1 * p1) + (2 * p2) + (3 * p3) + (4 * p4)

# # ==========================================
# # 1. BASELINE 1: NAIVE LMM
# # ==========================================
# print("Fitting 1. Naive LMM...")
# targets_naive = {'Overall': ['Total Score'], 'Domain': list(domains.keys()), 'Individual Item': all_items}

# for level, t_list in targets_naive.items():
#     for target in t_list:
#         formula = f"Q('{target}') ~ {covariates}" if ' ' in target else f"{target} ~ {covariates}"
#         try:
#             model = smf.mixedlm(formula, data=df, groups=df["subject_id"]).fit(reml=False, disp=False)
#             expected = model.fittedvalues
#             # Enforce clinical boundaries for Naive LMM metrics
#             max_val = 48 if level == 'Overall' else (12 if level == 'Domain' else 4)
#             expected = np.clip(expected, 0, max_val)
#             record_metrics('1. Naive LMM', level, target, df[target], expected, is_item=(level=='Individual Item'))
#         except: pass

# # ==========================================
# # 2. BASELINE 2: IRT + LATENT LMM
# # ==========================================
# print("Fitting 2. IRT + Latent LMM...")
# for target_name, items in scenarios.items():
#     level = 'Overall' if target_name == 'Total Score' else 'Domain'
#     irt_data = df[items].astype(int).values.T + 1
#     irt_results = grm_mml(irt_data)
#     df['theta'] = irt_results['Ability']
    
#     try:
#         model = smf.mixedlm(f"theta ~ {covariates}", data=df, groups=df["subject_id"]).fit(reml=False, disp=False)
#         expected_agg = np.zeros(len(df))
        
#         for i, item in enumerate(items):
#             exp_item = get_irt_expected(model.fittedvalues, irt_results['Discrimination'][i], irt_results['Difficulty'][i])
#             expected_agg += exp_item
#             if level == 'Domain': # Capture items during domain loops
#                 record_metrics('2. IRT + LMM', 'Individual Item', item, df[item], exp_item, is_item=True)
                
#         record_metrics('2. IRT + LMM', level, target_name, df[target_name], expected_agg, is_item=False)
#     except: pass

# ==========================================
# 3. BASELINE 3: DISCRETE-TIME FA
# ==========================================
print("Fitting 3. Discrete-Time FA...")
df = df.sort_values(by=['subject_id', 'ALSFRS_Delta'])
df['Delta_Lag'] = df.groupby('subject_id')['ALSFRS_Delta'].shift(1)
df['Time_Since_Last_Visit'] = df['ALSFRS_Delta'] - df['Delta_Lag']

for target_name, items in scenarios.items():
    level = 'Overall' if target_name == 'Total Score' else 'Domain'
    fa = FactorAnalyzer(n_factors=1, rotation=None)
    fa.fit(df[items])
    df['Factor'] = fa.transform(df[items])[:, 0]
    
    df['Factor_Lag'] = df.groupby('subject_id')['Factor'].shift(1)
    df_dyn = df.dropna(subset=['Factor_Lag', 'Time_Since_Last_Visit']).copy()
    formula = f"Factor ~ Factor_Lag + {covariates_ar}"
    
    try: # LMM with OLS Fallback
        try:
            model = smf.mixedlm(formula, data=df_dyn, groups=df_dyn["subject_id"])
            fitted_factor = model.fit(method='lbfgs', reml=False, disp=False).fittedvalues
        except:
            fitted_factor = smf.ols(formula, data=df_dyn).fit().fittedvalues
            
        expected_agg = np.zeros(len(df_dyn))
        for i, item in enumerate(items):
            exp_item = np.clip((fitted_factor * fa.loadings_[:, 0][i]) + df_dyn[items].mean().values[i], 0, 4)
            expected_agg += exp_item
            if level == 'Domain':
                record_metrics('3. Discrete FA', 'Individual Item', item, df_dyn[item], exp_item, is_item=True)
                
        record_metrics('3. Discrete FA', level, target_name, df_dyn[target_name], expected_agg, is_item=False)
    except: pass

# ==========================================
# 4. FORMAT AND PRINT OUTPUT
# ==========================================
eval_df = pd.DataFrame(results_list)
eval_df['Level'] = pd.Categorical(eval_df['Level'], categories=['Overall', 'Domain', 'Individual Item'], ordered=True)

# Helper to print pivot tables
def print_metric_table(metric_name, format_str):
    pivot = eval_df.pivot(index=['Level', 'Target'], columns='Model', values=metric_name)
    print("\n" + "="*80)
    print(f"BASELINE COMPARISON: {metric_name}")
    print("="*80)
    print(pivot.to_string(float_format=lambda x: format_str.format(x) if pd.notnull(x) else "-"))

print_metric_table('RMSE', "{:.3f}")
print_metric_table('MAE', "{:.3f}")
print_metric_table('Exact Match %', "{:.1f}%")
print_metric_table('Weighted Kappa', "{:.3f}")

Loading data and setting up...
Fitting 3. Discrete-Time FA...

BASELINE COMPARISON: RMSE
Model                                          3. Discrete FA
Level           Target                                       
Overall         Total Score                             3.557
Domain          Bulbar                                  1.013
                Fine_Motor                              1.402
                Gross_Motor                             1.449
                Respiratory                             1.287
Individual Item Q1_Speech                               0.576
                Q2_Salivation                           0.571
                Q3_Swallowing                           0.525
                Q4_Handwriting                          0.708
                Q5_Cutting                              0.638
                Q6_Dressing_and_Hygiene                 0.744
                Q7_Turning_in_Bed                       0.871
                Q8_Walking                 